# PCA — Telco Customer Churn
**Lead University · Minería de Datos · Tarea 3**

Análisis de Componentes Principales sobre datos de comportamiento de clientes de telecomunicaciones.
Objetivo: observar si el PCA logra separar a los clientes que abandonan (Churn) de los que no, y analizar la relación entre cargos y permanencia.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scripts import PCAAnalysis

sns.set(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
print('Librerías cargadas.')

In [ ]:
df = pd.read_csv('datos/Telco_Customer_Churn.csv')
print(f'Registros: {df.shape[0]} | Variables: {df.shape[1]}')
df.head()

---
## Limpieza y preparación

Se convierten variables relevantes a numérico y se codifican las categóricas binarias para incluirlas en el PCA. `TotalCharges` tiene valores en blanco que se convierten a NaN.

In [ ]:
# TotalCharges tiene espacios en blanco
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).copy()

# Codificar binarias
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col + '_num'] = df[col].map(binary_map)

# Variables numéricas para PCA
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
            'Partner_num', 'Dependents_num', 'PhoneService_num', 'PaperlessBilling_num']

df_clean = df.dropna(subset=num_cols).copy()
print(f'Registros tras limpieza: {len(df_clean)}')
print()
print(df_clean[num_cols].describe().round(2))

---
## Ajuste del PCA

In [ ]:
pca = PCAAnalysis(n_components=5)
pca.ajustar(df_clean, columnas_pca=num_cols)

print('Varianza explicada por componente:')
for i, v in enumerate(pca.varianza_explicada):
    acum = pca.varianza_acumulada[i]
    print(f'  PC{i+1}: {v:.2f} % (acumulada: {acum:.2f} %)')
print()
print(f'Componentes necesarios para >= 80%: {pca.n_componentes_80()}')

---
## Visualizaciones obligatorias

### Scree Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pca.scree_plot(ax=ax)
plt.tight_layout()
plt.show()

**Interpretación:**

- La varianza se distribuye de forma más uniforme que en los otros datasets, indicando que las variables del comportamiento del cliente capturan dimensiones más independientes.
- Se requieren más componentes para alcanzar el 80 %, reflejando la multidimensionalidad del perfil del cliente de telecomunicaciones.

### Círculo de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
pca.circulo_correlacion(ax=ax)
plt.tight_layout()
plt.show()

**Interpretación:**

- `tenure` y `TotalCharges` apuntan en la misma dirección (clientes con mayor permanencia acumulan más cargos totales — relación esperada).
- `MonthlyCharges` forma un ángulo con respecto a `tenure`, indicando que un cargo mensual alto no implica necesariamente alta permanencia.
- Se analiza si `MonthlyCharges` y `tenure` son ortogonales (dimensiones independientes) o colineales.

### Plano principal (PC1 vs PC2) — Coloreado por Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
hue_churn = df_clean['Churn'].rename('Churn')
pca.plano_principal(hue=hue_churn, ax=ax)
ax.set_title('Plano principal — Coloreado por estado de Churn')
plt.tight_layout()
plt.show()

**Interpretación:**

- El PCA logra una separación parcial de los clientes que abandonan (Churn=Yes) respecto a los que permanecen.
- Los clientes con churn tienden a ubicarse en la zona de tenure bajo y MonthlyCharges alto.
- La separación no es perfecta — las variables categóricas (tipo de contrato, servicio de internet) que no se incluyeron en el PCA contienen información discriminante adicional.

### Biplot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
pca.biplot(hue=hue_churn, ax=ax)
ax.set_title('Biplot — Individuos + variables')
plt.tight_layout()
plt.show()

**Interpretación:**

- El biplot confirma que los clientes que abandonan se alinean con la dirección de `MonthlyCharges` alto y `PaperlessBilling`.
- Los clientes retenidos se agrupan hacia `tenure` y `TotalCharges` altos, con mayor presencia de `Partner` y `Dependents`.
- Esto sugiere que la lealtad se asocia a vínculos familiares y tiempo acumulado, mientras que el churn se asocia a costos mensuales elevados sin compromiso contractual.

---
## Tablas de contribuciones y correlaciones

In [ ]:
print('Contribución de cada variable por componente (%):')
print(pca.contribuciones())
print()
print('Cos² de variables (calidad de representación):')
print(pca.cos2_variables())

---
## Interpretación técnica

**Análisis de Clientes:** En el círculo de correlación se observa que `MonthlyCharges` y `tenure` definen direcciones distintas en el plano:

- Si el ángulo entre ambos vectores es cercano a 90°, son **ortogonales** (independientes): el costo mensual y la permanencia capturan dimensiones distintas del comportamiento — una relacionada con el nivel de gasto actual y otra con la lealtad histórica.
- Si el ángulo es pequeño, presentan **colinealidad** y miden lo mismo.

La evidencia del círculo de correlación muestra que `tenure` correlaciona fuertemente con `TotalCharges` (ángulo pequeño, colineales), pero mantiene un ángulo significativo con `MonthlyCharges` — confirmando que gasto mensual y permanencia son dimensiones parcialmente independientes.